In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("ML libraries imported successfully!")

ML libraries imported successfully!


In [3]:
import urllib.request

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

urllib.request.urlretrieve(url, "spam_dataset.tsv")

print("Dataset downloaded successfully!")


Dataset downloaded successfully!


In [4]:
data = pd.read_csv(
    "spam_dataset.tsv",
    sep="\t",
    header=None,
    names=["label", "message"]
)

data.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
print("Dataset shape:", data.shape)

Dataset shape: (5572, 2)


In [6]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


In [7]:
print(data["label"].value_counts())

label
ham     4825
spam     747
Name: count, dtype: int64


In [8]:
data.isnull().sum()

,0
label,0
message,0


In [9]:
print("Duplicate rows:", data.duplicated().sum())

Duplicate rows: 403


In [10]:
data = data.drop_duplicates().reset_index(drop=True)

print("Dataset shape after removing duplicates:", data.shape)

Dataset shape after removing duplicates: (5169, 2)


In [11]:
data["label"] = data["label"].map({"ham": 0, "spam": 1})

print(data.head())

   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


In [12]:
X = data["message"]
y = data["label"]

print("Input data shape:", X.shape)
print("Target data shape:", y.shape)

Input data shape: (5169,)
Target data shape: (5169,)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

Training messages: 4135
Testing messages: 1034


In [14]:
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training data shape:", X_train_tfidf.shape)
print("Testing data shape:", X_test_tfidf.shape)

Training data shape: (4135, 7585)
Testing data shape: (1034, 7585)


In [15]:
model = MultinomialNB()

In [16]:
model.fit(X_train_tfidf, y_train)

print("Model training completed!")

Model training completed!


In [17]:
y_pred = model.predict(X_test_tfidf)

print("Predictions completed!")

Predictions completed!


In [18]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy percentage:", round(accuracy * 100, 2), "%")

Accuracy: 0.9584139264990329
Accuracy percentage: 95.84 %


In [19]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      1.00      0.98       894
           1       1.00      0.69      0.82       140

    accuracy                           0.96      1034
   macro avg       0.98      0.85      0.90      1034
weighted avg       0.96      0.96      0.96      1034



In [20]:
from sklearn.metrics import classification_report

In [21]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      1.00      0.98       894
           1       1.00      0.69      0.82       140

    accuracy                           0.96      1034
   macro avg       0.98      0.85      0.90      1034
weighted avg       0.96      0.96      0.96      1034



In [22]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[894   0]
 [ 43  97]]


In [23]:
def check_message(message):
    message_tfidf = vectorizer.transform([message])
    prediction = model.predict(message_tfidf)[0]

    if prediction == 1:
        return "Spam"
    else:
        return "Ham"

In [24]:
test_message = "Congratulations! You have won a free prize. Click here to claim it."
print("Prediction:", check_message(test_message))

Prediction: Spam


In [25]:
test_message = "Hi, are you coming to college tomorrow?"
print("Prediction:", check_message(test_message))

Prediction: Ham


In [26]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 61.3 MB/s eta 0:00:00


In [27]:
%%writefile app.py

import streamlit as st

st.title("Spam Email Detector")

message = st.text_area("Enter your message:")

if st.button("Check Message"):
    if message.strip() == "":
        st.warning("Please enter a message.")
    else:
        result = check_message(message)

        if result == "Spam":
            st.error("This message is Spam")
        else:
            st.success("This message is Ham")

Writing app.py


In [28]:
import joblib

joblib.dump(model, "spam_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [29]:
%%writefile app.py

import streamlit as st
import joblib

model = joblib.load("spam_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

st.title("Spam Email Detector")

message = st.text_area("Enter your message:")

if st.button("Check Message"):
    if message.strip() == "":
        st.warning("Please enter a message.")
    else:
        message_tfidf = vectorizer.transform([message])
        prediction = model.predict(message_tfidf)[0]

        if prediction == 1:
            st.error("This message is Spam")
        else:
            st.success("This message is Ham")

Overwriting app.py


In [30]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [31]:
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://twenty-candies-eat.loca.lt
^C


In [32]:
%%writefile app.py

import streamlit as st
import joblib

model = joblib.load("spam_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

st.title("Spam Email Detector")

message = st.text_input("Enter your message:")

if st.button("Check Message"):
    if message.strip() == "":
        st.warning("Please enter a message.")
    else:
        message_tfidf = vectorizer.transform([message])
        prediction = model.predict(message_tfidf)[0]

        if prediction == 1:
            st.error("This message is Spam")
        else:
            st.success("This message is Ham")

Overwriting app.py


In [33]:
!pkill -f streamlit
!pkill -f localtunnel

In [34]:
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼your url is: https://slick-papers-sniff.loca.lt
^C


In [35]:
!cat /content/logs.txt

In [36]:
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼your url is: https://cuddly-gifts-guess.loca.lt
^C


In [37]:
!pkill -f streamlit
!pkill -f localtunnel

In [38]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

In [39]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8501)")
print(url)

https://8501-m-s-kkb-use5b1-y8wxdmcv54kk-b.us-east5-1.prod.colab.dev


In [40]:
!cat /content/streamlit.log



2026-10-06 10:29:45.258 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.162.64.106:8501

2026-10-06 10:30:17.332 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-use5b1-y8wxdmcv54kk-b.us-east5-1.prod.colab.dev, host=m-s-kkb-use5b1-y8wxdmcv54kk.us-east5-b.c.codatalab-user-runtimes.internal:8007
2026-10-06 10:30:20.421 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-use5b1-y8wxdmcv54kk-b.us-east5-1.prod.colab.dev, host=m-s-kkb-use5b1-y8wxdmcv54kk.us-east5-b.c.codatalab-user-runtimes.internal:8007
2026-10-06 10:30:23.095 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-use5b1-y8wxdmcv54kk-b.us-east5-1.prod.colab.dev, host=m-s-kkb-use5b1-y8wxdmcv54kk.us-east5-b.c.codatalab-user-runtimes.internal:8

In [41]:
!ps aux | grep streamlit

root        1560  1.5  0.5 1230592 74152 ?       Sl   10:29   0:05 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8501 --server.address 0.0.0.0
root        2904  0.0  0.0   7344  3668 ?        S    10:35   0:00 /bin/bash -c ps aux | grep streamlit
root        2906  0.0  0.0   6548  2300 ?        S    10:35   0:00 grep streamlit


In [42]:
!pkill -f streamlit

In [43]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit.log 2>&1 &

In [44]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8501)")
print(url)

https://8501-m-s-kkb-use5b1-y8wxdmcv54kk-b.us-east5-1.prod.colab.dev


In [45]:
from google.colab import files

files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
files.download("spam_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
files.download("tfidf_vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
files.download("spam_dataset.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [49]:
from google.colab import files
files.download("Spam_Email_Detection.ipynb")

FileNotFoundError: Cannot find file: Spam_Email_Detection.ipynb